# MedNote MCP & EHR Tools — Concepts + Step-by-Step (Tasks 10–13)

This notebook explains the tool system on `ft_mcp_build` from first principles:
**what MCP is**, how its **client–server architecture** works, **how an LLM
decides to call a tool**, and how our **mock EHR** maps onto a real deployment
— then proves every layer live, ending with a genuine MCP handshake over
stdio JSON-RPC. Companion doc: [`docs/mcp_tools_notes.md`](../mcp_tools_notes.md).

**Part A — Concepts (read, no code):**

| § | What |
|---|------|
| 1 | What is MCP, and what problem does it solve? |
| 2 | The client–server architecture & session lifecycle |
| 3 | How the LLM decides to call a tool — request flow diagrams |
| 4 | Mock vs. real: how this server would be used in production |

**Part B — Hands-on (every layer, live):**

| § | What | Cost |
|---|------|------|
| 5 | Setup | free |
| 6 | Start the mock EHR (FastAPI) inside this kernel | free |
| 7 | The raw HTTP endpoints + response envelopes | free |
| 8 | `ehr_client` — the single integration point + its failure mode | free |
| 9 | The LangChain tool surface (what the agent's LLM sees) | free |
| 10 | FastMCP in-process: `tools/list` and `tools/call` | free |
| 11 | The real MCP wire: stdio JSON-RPC handshake | free |
| 12 | Optional: the agent's `tool_execution` node end-to-end | 💰 1 Gemini call |
| 13 | Shutdown + store cleanup | free |

> ⚠️ §6 binds the EHR port from `config.yml → ehr_api` (default **8100**). If
> you already have `uvicorn mednote.tools.ehr_api:app` running in a terminal,
> skip §6 — every other section works against either instance.


## 1 · What is MCP, and what problem does it solve?

**MCP (Model Context Protocol)** is an open standard (introduced by Anthropic,
Nov 2024) that gives LLM applications one uniform way to reach external
systems — files, databases, APIs, in our case an EHR.

### The M×N problem

Without a standard, every AI app must write custom glue for every system:

```
   M apps                custom glue               N systems
┌────────────┐      ╔═══════════════════╗      ┌─────────────┐
│ Claude     │──────╢                   ╟──────│ EHR         │
│ Desktop    │──────╢   M × N bespoke   ╟──────│ Database    │
│ IDE agent  │──────╢   integrations    ╟──────│ File system │
│ Our agent  │──────╢   (each written   ╟──────│ Ticketing   │
│ ...        │──────╢    twice, badly)  ╟──────│ ...         │
└────────────┘      ╚═══════════════════╝      └─────────────┘
```

MCP turns M×N into **M+N**: each app implements the protocol once (as a
*host/client*), each system is wrapped once (as a *server*), and any client
can then talk to any server. Think *USB-C for AI tools*: one connector shape,
many devices.

### What a server can expose (the three primitives)

| Primitive | Direction | Meaning | Used here? |
|-----------|-----------|---------|------------|
| **Tools** | model-controlled | Actions the LLM may *choose* to invoke (POST-like, side effects allowed) | ✅ `save_note`, `get_patient_history` |
| **Resources** | app-controlled | Read-only data the host can attach as context (GET-like) | ❌ (demographics could be one later) |
| **Prompts** | user-controlled | Reusable prompt templates the user picks | ❌ |

Under the hood everything is **JSON-RPC 2.0** messages — the same wire format
you'll see for real in §11.


## 2 · The client–server architecture & session lifecycle

MCP names three roles. The distinction that matters: the **host** is the AI
application the user talks to; it embeds one **client** per connected server;
each **server** wraps one backend and advertises its capabilities.

```
┌─────────────────────────── HOST ────────────────────────────┐
│  the AI application (Claude Desktop, Claude Code, an IDE,   │
│  or our LangGraph agent playing the same role in-process)   │
│                                                             │
│   LLM ◄──── tool schemas / tool call results                │
│    │                                                        │
│    ▼ proposed tool calls                                    │
│  ┌──────────────┐            ┌──────────────┐               │
│  │ MCP client 1 │            │ MCP client 2 │  (1 client    │
│  └──────┬───────┘            └──────┬───────┘   per server) │
└─────────┼───────────────────────────┼───────────────────────┘
          │ JSON-RPC 2.0              │
          │ (stdio or HTTP)           │
          ▼                           ▼
   ┌─────────────┐             ┌─────────────┐
   │ MCP server  │             │ MCP server  │
   │ mednote-ehr │             │ e.g. github │
   └──────┬──────┘             └──────┬──────┘
          │ any protocol the backend speaks (here: HTTP/REST)
          ▼
   ┌─────────────┐
   │  EHR system │  ← mocked by our FastAPI app (see §4)
   └─────────────┘
```

### Transports — how the bytes move

- **stdio** (ours): the host *spawns the server as a subprocess* and speaks
  JSON-RPC over its stdin/stdout. Zero network setup, credentials stay on the
  user's machine. This is what `claude mcp add mednote-ehr -- uv run python -m
  mednote.mcp.server` configures.
- **Streamable HTTP**: the server runs remotely; clients POST JSON-RPC over
  HTTP (with optional SSE streaming). Used for shared/hosted servers.

The protocol messages are identical either way — only the pipe differs.

### Session lifecycle (you will see each step live in §11)

```
client                                  server
  │  initialize(protocolVersion, ...)     │   1. handshake: agree on protocol
  │ ─────────────────────────────────────►│      version + capabilities
  │ ◄───────────────────────────────────── │      (server introduces itself)
  │  notifications/initialized            │
  │ ─────────────────────────────────────►│
  │                                       │
  │  tools/list                           │   2. discovery: client fetches the
  │ ─────────────────────────────────────►│      tool catalog — names, docs,
  │ ◄─────  [{name, description,          │      JSON Schemas for arguments
  │           inputSchema}, ...]          │
  │                                       │
  │  tools/call(name, arguments)          │   3. invocation: execute one tool,
  │ ─────────────────────────────────────►│      return content (+ errors as
  │ ◄─────  {content: [...], isError}     │      data, not crashes)
```

The schemas fetched in step 2 are what the host injects into its LLM's
context — which is exactly where the next section picks up.


## 3 · How the LLM decides to call a tool

The most important mental model in this notebook: **the LLM never executes
anything.** It only ever emits *text* or a *structured proposal* ("call tool X
with args Y"). The runtime around it — our `tool_execution` node, or an MCP
host — validates and executes the proposal, then feeds the result back.

What makes the model "decide" to call a tool? Three inputs sit in its context:

1. **The tool schemas** (from `bind_tools()` / `tools/list`) — name,
   description, and argument JSON Schema. The model reads these like API docs;
   *the description is prompt engineering* (ours carries the physician-
   confirmation rule inside it).
2. **A system prompt** stating the rules of engagement (`TOOL_SYSTEM_PROMPT`:
   pick ONE tool, never invent patient IDs or note content, decline if the
   request is incomplete).
3. **The user request + context** the runtime assembled.

### Flow diagram — a physician's request through the MedNote agent

```
Physician: "Save this note to the patient's chart."
    │
    ▼
[1] parse_input ─── keyword/dialogue classification ──► intent = "save"
    │
    │   ★ SAFETY GATE (guardrail G5): tool_execution is ONLY reachable via
    │     this explicit save intent — the graph topology itself prevents
    │     auto-saving. No LLM judgment involved in the gate.
    ▼
[2] tool_execution assembles the REQUEST CONTEXT from state:
        user_input  = "Save this note to the patient's chart."
        patient_id  = "P001"
        draft_note  = "### Subjective ..."        (if present)
        codes       = ["G44.2", ...]              (if present)
    │
    ▼
[3] get_tool_llm().invoke([...])       ┌──────────────────────────────────┐
        = main LLM + bind_tools ─────► │ LLM CONTEXT                      │
                                       │  system: TOOL_SYSTEM_PROMPT      │
                                       │  tools:  save_note(schema)       │
                                       │          get_patient_history(sch)│
                                       │  human:  request context [2]     │
                                       └────────────────┬─────────────────┘
    ┌───────────────────────────────────────────────────┤ the model chooses
    │                                                   │
    │ A. request complete & actionable                  │ B. something missing
    ▼                                                   ▼    (e.g. no draft note)
LLM emits STRUCTURED TOOL CALL                     LLM emits plain text
{"name": "save_note",                              "No draft note was provided
 "args": {"patient_id": "P001",                     to save."
          "note": "### Subjective ...",                 │
          "icd_codes": ["G44.2"]}}                      ▼
    │                                              ToolResult ok=False,
    ▼                                              honest reply — nothing
[4] RUNTIME executes (not the LLM!):               was written to the EHR
    registry["save_note"].invoke(args)
    │
    ▼
[5] ehr_client.post_note() ──HTTP──► POST /notes ──► EHR store
    │                                                    │
    │  EhrApiError (EHR down)?                           ▼
    │  → ToolResult ok=False,                {"status": "saved",
    │    "the note was NOT saved",            "note_id": "N_ab12cd34"}
    │    error on errors channel                         │
    ▼                                                    ▼
[6] envelope ──► typed ToolResult ──► response_generation
                                              │
                                              ▼
              Physician sees: "Note saved to patient P001's chart
                               (note ID N_ab12cd34)."
```

### The same decision, in a generic MCP host

When Claude Desktop (or any host) uses our server, the loop is identical —
only the plumbing differs:

```
session start:  client ──initialize──► server
                client ──tools/list──► server ──► schemas → injected into LLM context

user turn:      "Pull up P001's visit history"
                    │
                    ▼
                LLM sees schemas + request ──► emits tool_use block
                    │                          {name: get_patient_history,
                    ▼                           input: {patient_id: "P001"}}
                host asks the USER for permission   ← extra gate real hosts add
                    │ approved
                    ▼
                client ──tools/call──► server ──HTTP──► EHR ──► envelope
                    │◄─────────────────result──────────────────────┘
                    ▼
                result appended to LLM context ──► model writes final answer
```

Same three beats every time: **schemas in → structured proposal out →
runtime executes and feeds the result back.**


## 4 · Mock vs. real: how this server would be used in production

We mock the EHR because requirements §4 forbid touching a real one — but the
mock is shaped so that *swapping it for the real thing changes exactly one
module*. That is the point of the `ehr_client` seam.

| Layer | This project (demo) | Real-time deployment |
|-------|--------------------|---------------------|
| EHR backend | FastAPI + `data/ehr_store.json`, seeded synthetic patients | Hospital EHR (e.g. Epic, Oracle Health) exposing **FHIR R4** REST APIs |
| Save a note | `POST /notes` | `POST /DocumentReference` (FHIR) |
| Demographics | `GET /patients/{id}` | `GET /Patient/{id}` (FHIR) |
| Visit history | `GET /patients/{id}/history` | `GET /Encounter?patient={id}` / Composition search |
| Auth | none (localhost) | **SMART-on-FHIR / OAuth 2.0** — the MCP server holds credentials, tokens never reach the LLM |
| Audit | JSON store + traces (Task 24) | EHR-side audit log **plus** our trace files (requirements §5.6) |
| Data | synthetic, no PHI | real PHI → HIPAA: TLS, minimum-necessary scopes, BAAs |
| MCP transport | stdio (local subprocess) | stdio for clinician desktops; **streamable HTTP** for a shared hospital-hosted server |

**What stays identical in production** — and this is the architectural payoff:

- The MCP surface: tool names, descriptions, argument schemas. Any MCP host
  already configured against `mednote-ehr` keeps working untouched.
- The agent: `tool_execution`, `TOOL_SYSTEM_PROMPT`, the G5 routing gate,
  the typed `ToolResult` handling.
- The failure contract: backend trouble → `EhrApiError` → "the note was NOT
  saved", never a stack trace, never a silent failure.

Only `ehr_client.py` (base URL, auth headers, FHIR payload mapping) and the
server-side config change. The LLM never learns the EHR moved.

**Where the pieces run in a real clinic:**

```
clinician's workstation                      hospital network
┌───────────────────────────────┐          ┌──────────────────────┐
│ MCP host (Claude Desktop /    │  OAuth2  │  FHIR gateway        │
│ hospital AI workspace)        │  + TLS   │  (Epic / Oracle ...) │
│   └─ MCP client ── stdio ──►  │          │          ▲           │
│       mednote-ehr MCP server ─┼──HTTPS───┼──────────┘           │
│       (holds the credentials) │          │   audit log, consent │
└───────────────────────────────┘          └──────────────────────┘
```

Everything below this line runs that architecture live — against the mock.


## 5 · Setup

Same defensive setup as the other notebooks: repo-root discovery, `.env`
loaded with `override=True`, config cache cleared so a long-lived kernel
re-reads `config.yml`. The API key only matters for §12.


In [5]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "config.yml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

import os
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(override=True)

from mednote.config import get_config
get_config.cache_clear()
cfg = get_config()
BASE = f"http://{cfg.ehr_api.host}:{cfg.ehr_api.port}"
print("EHR base URL :", BASE)
print("store path   :", cfg.paths.ehr_store_path)
print("main LLM     :", cfg.llm.provider, cfg.llm.model, "(only used in §12)")

EHR base URL : http://localhost:8100
store path   : data/ehr_store.json
main LLM     : google gemini-pro-latest (only used in §12)


## 6 · Start the mock EHR inside this kernel

`ehr_api.py` is a plain FastAPI app, so we can run uvicorn in a **daemon
thread** — no separate terminal needed. `server.should_exit = True` in §13
shuts it down cleanly.

On first write it creates `data/ehr_store.json` (gitignored) from
`SEED_PATIENTS` — the P001–P013 patients of the synthetic transcript set plus
the UI's `P-DEMO`, so demographics always match the Task 4 eval labels.

> In the §4 production picture, this cell is the piece that disappears: the
> hospital's FHIR gateway is already running. Everything after this cell
> would stay the same.


In [8]:
import threading
import time

import httpx
import uvicorn

server = uvicorn.Server(uvicorn.Config(
    "mednote.tools.ehr_api:app",
    host=cfg.ehr_api.host, port=cfg.ehr_api.port, log_level="warning",
))
ehr_thread = threading.Thread(target=server.run, daemon=True)
ehr_thread.start()

for _ in range(50):
    try:
        httpx.get(f"{BASE}/patients/P001", timeout=0.5)
        break
    except httpx.HTTPError:
        time.sleep(0.2)
else:
    raise RuntimeError(f"EHR did not come up on {BASE} — port already in use?")

print(f"Mock EHR serving at {BASE}")

ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8100): only one usage of each socket address (protocol/network address/port) is normally permitted


Mock EHR serving at http://localhost:8100


## 7 · The raw HTTP endpoints

Three endpoints, one consistent envelope family (`status` is always present:
`saved` / `found` / `no_history` / `error`). Note the two failure shapes:
unknown patient → **404 error envelope**, invalid payload → **422** from
pydantic validation *before* anything touches the store.

This layer is EHR-speak, not MCP-speak — the MCP server (§10–11) will wrap
it so an LLM host never needs to know these URLs exist.


In [9]:
import json

def show(label: str, resp: httpx.Response) -> None:
    print(f"{label}  [{resp.status_code}]")
    print(json.dumps(resp.json(), indent=2)[:400])
    print()

with httpx.Client(base_url=BASE, timeout=5) as c:
    show("GET  /patients/P005           ", c.get("/patients/P005"))
    show("POST /notes                   ", c.post("/notes", json={
        "patient_id": "P001",
        "note": "### Subjective\nHeadache for 3 days, worse in the morning...",
        "icd_codes": ["G44.2"],
    }))
    show("GET  /patients/P001/history   ", c.get("/patients/P001/history"))
    show("GET  /patients/NOPE  (404)    ", c.get("/patients/NOPE"))
    show("POST /notes empty note (422)  ", c.post("/notes", json={
        "patient_id": "P001", "note": "",
    }))

GET  /patients/P005             [200]
{
  "status": "found",
  "patient": {
    "patient_id": "P005",
    "name": "Aarav Patel",
    "age": 4,
    "sex": "male"
  }
}

POST /notes                     [200]
{
  "status": "saved",
  "note_id": "N_1e7dffac",
  "patient_id": "P001",
  "timestamp": "2026-07-19T11:36:05.361338+00:00"
}

GET  /patients/P001/history     [200]
{
  "status": "found",
  "patient_id": "P001",
  "visits": [
    {
      "note_id": "N_30026586",
      "patient_id": "P001",
      "date": "2026-07-18",
      "note": "Smoke-test SOAP note",
      "icd_codes": [
        "G44.2"
      ],
      "created_at": "2026-07-18T13:43:50.937040+00:00"
    },
    {
      "note_id": "N_0f54d480",
      "patient_id": "P001",
      "date": "2026-07-19",
      "

GET  /patients/NOPE  (404)      [404]
{
  "status": "error",
  "message": "Unknown patient ID 'NOPE'."
}

POST /notes empty note (422)    [422]
{
  "detail": [
    {
      "type": "string_too_short",
      "loc": [
        "bod

## 8 · `ehr_client` — the single integration point

Every consumer goes through these three functions; none of them build URLs or
parse envelopes anywhere else. `get_client()` is the deliberate seam — tests
monkeypatch it to return a FastAPI `TestClient`, and a **production
deployment swaps it for an authenticated FHIR client** (§4) without touching
any tool or agent code.

The one failure contract: any connection/protocol problem raises
`EhrApiError`. Callers must degrade gracefully — the agent turns it into
*"the note was NOT saved"* on the errors channel, never a stack trace.


In [10]:
from mednote.tools import ehr_client

print("demographics:", ehr_client.fetch_demographics("P005")["patient"])
saved = ehr_client.post_note("P001", "Saved via ehr_client directly.", ["G44.2"])
print("post_note   :", saved["status"], saved["note_id"])
print("history     :", len(ehr_client.fetch_history("P001")["visits"]), "visit(s)")

# The failure mode: aim the injectable client at a dead port.
try:
    ehr_client.post_note(
        "P001", "x",
        client=httpx.Client(base_url="http://127.0.0.1:59999", timeout=0.3),
    )
except ehr_client.EhrApiError as exc:
    print("\nEhrApiError :", str(exc)[:120], "...")

demographics: {'patient_id': 'P005', 'name': 'Aarav Patel', 'age': 4, 'sex': 'male'}
post_note   : saved N_dd8dd8a8
history     : 8 visit(s)

EhrApiError : EHR API unreachable: timed out ...


## 9 · The LangChain tool surface

`@tool` turns each function into a `BaseTool` whose **signature + docstring
become the LLM-facing schema** — this is input #1 of the §3 decision loop,
exactly what `get_tool_llm()` (`llm.bind_tools([...])`) serves to Gemini in
the agent's `tool_execution` node.

Note the guardrail sentence in `save_note`'s description: the confirmation
rule travels *inside the schema* to every model that binds it. The
description is prompt engineering.


In [ ]:
from mednote.tools import get_patient_history, save_note

for t in (save_note, get_patient_history):
    print(f"── {t.name}")
    print("   args:", json.dumps(t.args))
    print("   doc :", t.description.strip().splitlines()[0])
print()

result = save_note.invoke({
    "patient_id": "P002",
    "note": "Saved via the LangChain tool surface.",
})
print("save_note.invoke          ->", result["status"], result["note_id"])
print("get_patient_history.invoke->",
      get_patient_history.invoke({"patient_id": "P002"})["status"])

## 10 · FastMCP in-process: `tools/list` and `tools/call`

`mcp/server.py` wraps the same two operations in **FastMCP** (the decorator
API of the official `mcp` SDK — the low-level `Server` class has no `@tool`
decorator). FastMCP generates each tool's JSON Schema from the Python type
hints and docstring: compare the output below with §9 — **same contract,
different protocol**. That's the M+N promise from §1 made concrete.

The server object is just Python, so before touching any transport we can
call the MCP operations directly (Jupyter allows top-level `await`). This is
lifecycle step 2 (discovery) and step 3 (invocation) from §2 — minus the wire.


In [11]:
from mednote.mcp.server import mcp as mcp_server

tools = await mcp_server.list_tools()
for t in tools:
    print(f"── {t.name}")
    print(json.dumps(t.inputSchema, indent=2))
    print()

── save_note
{
  "properties": {
    "patient_id": {
      "title": "Patient Id",
      "type": "string"
    },
    "note": {
      "title": "Note",
      "type": "string"
    },
    "icd_codes": {
      "anyOf": [
        {
          "items": {
            "type": "string"
          },
          "type": "array"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Icd Codes"
    }
  },
  "required": [
    "patient_id",
    "note"
  ],
  "title": "save_noteArguments",
  "type": "object"
}

── get_patient_history
{
  "properties": {
    "patient_id": {
      "title": "Patient Id",
      "type": "string"
    }
  },
  "required": [
    "patient_id"
  ],
  "title": "get_patient_historyArguments",
  "type": "object"
}



In [ ]:
result = await mcp_server.call_tool("save_note", {
    "patient_id": "P003",
    "note": "Saved through MCP call_tool (in-process).",
})
print(result)

## 11 · The real MCP wire: stdio JSON-RPC

Now the genuine article — the full §2 lifecycle over a real pipe. An **MCP
client** spawns `python -m mednote.mcp.server` as a subprocess and speaks
JSON-RPC over its stdin/stdout: the exact mechanics Claude Desktop / Claude
Code use after
`claude mcp add mednote-ehr -- uv run python -m mednote.mcp.server`.

Watch the output map 1:1 onto the §2 sequence diagram: `initialize`
(handshake + server identity + protocol version) → `tools/list` (discovery)
→ `tools/call` (invocation).

We run the client in a *separate process* (`subprocess.run`) because asyncio
subprocess support inside a Jupyter kernel on Windows is unreliable (the
kernel's selector event loop can't spawn processes). The loop this creates
is worth tracing:

```
this kernel ─spawns─► MCP client ─spawns─► MCP server ─HTTP─► EHR thread
                      (stdio JSON-RPC)                        (§6, this kernel!)
```


In [ ]:
import subprocess
import sys

client_script = '''
import asyncio, sys

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    params = StdioServerParameters(command=sys.executable, args=["-m", "mednote.mcp.server"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            info = await session.initialize()
            print("initialize  ->", info.serverInfo.name, "protocol", info.protocolVersion)

            tools = await session.list_tools()
            print("tools/list  ->", [t.name for t in tools.tools])

            result = await session.call_tool(
                "save_note",
                {"patient_id": "P001", "note": "Saved over the real MCP stdio wire."},
            )
            print("tools/call  ->", result.content[0].text)

asyncio.run(main())
'''

proc = subprocess.run(
    [sys.executable, "-c", client_script],
    capture_output=True, text=True, timeout=120, cwd=REPO_ROOT,
)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr[-1500:])

## 12 · Optional: the agent's `tool_execution` node 💰

The §3 flow diagram, executed for real. `tool_execution` is reached only on
an explicit `save` intent — **the routing itself is guardrail G5** (no
auto-save without confirmation). The node builds a request context from
state, invokes `get_tool_llm()` (main LLM + `bind_tools`) under
`TOOL_SYSTEM_PROMPT`, executes the chosen tool call, and maps the envelope
onto a typed `ToolResult`.

Here we call the node directly with a hand-built state carrying a draft note
— 💰 **1 Gemini call**. Then try branch B of the §3 diagram: remove
`draft_note` and re-run — the prompt forbids inventing content, so the LLM
should make *no* tool call and the node reports what's missing instead.


In [ ]:
from mednote.agent import nodes

update = nodes.tool_execution({
    "user_input": "Save this note to the patient's chart.",
    "patient_id": "P001",
    "draft_note": (
        "### Subjective\nHeadache for 3 days, worse in the morning.\n"
        "### Objective\nBP 130/85.\n"
        "### Assessment\nPossible tension-type headache - FOR PHYSICIAN REVIEW.\n"
        "### Plan\nFollow-up in 2 weeks."
    ),
    "suggested_codes": [{"code": "G44.2"}],
    "trace_id": "nb-mcp-demo",
    "errors": [],
})

print("ok     :", update["tool_result"]["ok"])
print("note_id:", update["tool_result"]["note_id"])
print("detail :", update["tool_result"]["detail"])

## 13 · Shutdown + store cleanup

Stop the §6 uvicorn thread. The EHR holds no Qdrant-style lock — the JSON
store is opened per request — so this is just tidiness. The saved notes stay
in `data/ehr_store.json` (gitignored); delete the file to reset the store to
its seed patients.


In [7]:
server.should_exit = True
ehr_thread.join(timeout=5)
print("EHR server stopped:", not ehr_thread.is_alive())

# To reset the store to seed data, uncomment:
# Path(cfg.paths.ehr_store_path).unlink(missing_ok=True)
# print("Store reset - next request re-seeds P001-P013 + P-DEMO.")

EHR server stopped: True
